# 多クラス分類と多ラベル分類（評価指標と訓練 CV による閾値選択）

書籍 第2章「分類・回帰問題の解き方」（2-2 文書分類：2-2-1 タスク、練習問題 2）に対応する演習ノートブックです。
CPU のみ・数秒で実行でき、1 つの文書に複数のカテゴリが同時に付きうる**多ラベル分類（multi-label classification）**の仕組みと評価尺度を体験します。

1. **重複なしデータでの訓練・交差検証**: 24 件の重複しないニュース見出しで 4 分割 CV を実施し、Micro-F1 を最大化する閾値を選択
2. **多クラス分類（Top-1）vs 多ラベル分類の比較**: 標準閾値 `0.50` と訓練 CV 選択閾値をテストデータで比較
3. **評価指標の違い**: Subset Accuracy（完全一致）と Hamming Loss / Micro-F1 / Macro-F1 / Samples-F1 の違い

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, hamming_loss
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MultiLabelBinarizer

---

## 1. サンプルデータと訓練 CV による閾値選択

書籍 2-2-1（p.59）の「元オリンピック選手が参議院に立候補する → スポーツ＋政治」のように、ニュース記事には複数のラベルが同時に付くことがあります。
ここでは `政治`・`経済`・`スポーツ`・`IT` の 4 カテゴリをバランスよく含む **重複のない 24 件の訓練見出し** と **8 件のテスト見出し** を用意し、テストデータを見ずに **訓練データの 4 分割交差検証（`cross_val_predict`）** だけで最適な出力閾値 `best_th` を選びます（同一事例を複数回複製すると CV でフォールド間リークが起きるため、すべて異なる見出しを使います）。

In [2]:
train_samples = [
    ("首相が来年度の税制改正案と経済対策について国会で所信表明を行った", ["政治", "経済"]),
    ("中央銀行が政策金利の据え置きを決定し、円相場と株価が反応した", ["経済"]),
    ("与党と野党が補正予算案の審議日程をめぐって協議を続けた", ["政治"]),
    ("生成AIの規制に関する新たな政府ガイドラインと法案が国会に提出された", ["政治", "IT"]),
    ("大手IT企業がクラウド基盤と半導体製造へ巨額の設備投資を発表した", ["経済", "IT"]),
    ("元オリンピック金メダリストが夏の参議院選挙への立候補を表明した", ["政治", "スポーツ"]),
    ("プロ野球の開幕戦が満員の球場で開催され新人が決勝打を放った", ["スポーツ"]),
    ("スポーツ庁がAIを活用した選手データ分析の推進事業を予算化した", ["政治", "スポーツ", "IT"]),
    ("国内スタートアップが新しい大規模言語モデルの商用APIを公開した", ["経済", "IT"]),
    ("サッカー日本代表の国際親善試合でスポンサー企業の株価が上昇した", ["経済", "スポーツ"]),
    ("政府がデジタル庁のシステム刷新に向けたサイバーセキュリティ対策を強化", ["政治", "IT"]),
    ("大手自動車メーカーが自動運転ソフトウェア開発企業を買収し決算が増益", ["経済", "IT"]),
    ("陸上世界選手権の代表選考会が国立競技場で開かれ日本新記録が出た", ["スポーツ"]),
    ("自治体がプロスポーツチームと連携した地域経済活性化策を発表した", ["政治", "経済", "スポーツ"]),
    ("新興テック企業の資金調達額が前年比2倍となり市場の注目を集めた", ["経済", "IT"]),
    ("元プロサッカー選手がスポーツ振興担当の議員連盟で講演した", ["政治", "スポーツ"]),
    ("ソフトウェア開発企業各社がクラウド基盤と生成AIの新サービスを公開した", ["IT"]),
    ("衆議院の総選挙に向けた各党の公約と国会での外交法案が議論された", ["政治"]),
    ("政府が半導体投資と物価対策に向けた新たな経済支援策を決定した", ["政治", "経済"]),
    ("プロ野球とサッカーの新規スポンサー契約により球団経営の収益が拡大した", ["経済", "スポーツ"]),
    ("IT企業が競技用AIカメラとデータ配信システムを全国大会へ導入した", ["スポーツ", "IT"]),
    ("政府が行政システムのクラウド移行とサイバーセキュリティ法案を提出した", ["政治", "IT"]),
    ("企業の決算発表で半導体や自動車株が買われ市場の株価が上昇した", ["経済"]),
    ("世界水泳の代表選考会で日本選手団が新記録を出し五輪代表に決まった", ["スポーツ"]),
]

test_samples = [
    ("元五輪選手が国会でスポーツ予算と経済効果について質問に立った", ["政治", "経済", "スポーツ"]),
    ("政府がAIスタートアップへの補助金と規制緩和の新政策を決定した", ["政治", "経済", "IT"]),
    ("国内IT大手がプロ野球球団の放映権ビジネスで増収を達成した", ["経済", "スポーツ", "IT"]),
    ("サイバー攻撃対策の新法案が参議院本会議で可決された", ["政治", "IT"]),
    ("半導体大手の四半期決算が過去最高益となり市場で買いが広がった", ["経済", "IT"]),
    ("プロ野球とサッカー日本代表の国際試合が国立競技場で開催された", ["スポーツ"]),
    ("衆議院本会議で新年度予算案の採決と物価高対策が議論された", ["政治", "経済"]),
    ("IT各社が新しい生成AI基盤モデルとクラウド開発環境を発表した", ["IT"]),
]

mlb = MultiLabelBinarizer(classes=["政治", "経済", "スポーツ", "IT"])
train_texts = [txt for txt, _ in train_samples]
test_texts = [txt for txt, _ in test_samples]
Y_train = mlb.fit_transform([lbls for _, lbls in train_samples])
Y_test = mlb.transform([lbls for _, lbls in test_samples])

print("訓練データ件数:", len(train_samples), "| ラベル出現数:", dict(zip(mlb.classes_, Y_train.sum(axis=0).tolist())))
print("テストデータ件数:", len(test_samples), "| ラベル出現数:", dict(zip(mlb.classes_, Y_test.sum(axis=0).tolist())))

# パイプライン化することで CV の各フォールド内で TF-IDF を学習し、リークを防ぐ
pipe = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)),
    OneVsRestClassifier(LogisticRegression(max_iter=1000, C=5.0)),
)

cv = KFold(n_splits=4, shuffle=True, random_state=42)
oof_probs = cross_val_predict(pipe, train_texts, Y_train, cv=cv, method="predict_proba")

candidates = np.round(np.arange(0.20, 0.61, 0.05), 2)
cv_rows = []
for th in candidates:
    Y_oof = (oof_probs >= th).astype(int)
    cv_rows.append({
        "閾値": float(th),
        "訓練CV Micro-F1": round(f1_score(Y_train, Y_oof, average="micro", zero_division=0), 3),
        "訓練CV Samples-F1": round(f1_score(Y_train, Y_oof, average="samples", zero_division=0), 3),
    })

cv_df = pd.DataFrame(cv_rows)
best_th = float(cv_df.loc[cv_df["訓練CV Micro-F1"].idxmax(), "閾値"])
print(f"\n4-fold CV で選ばれた閾値: best_th = {best_th:.2f}")
display(cv_df)

# 全訓練データで再学習し、テストデータの予測確率を得る
pipe.fit(train_texts, Y_train)
probs = pipe.predict_proba(test_texts)

訓練データ件数: 24 | ラベル出現数: {'政治': 11, '経済': 11, 'スポーツ': 10, 'IT': 10}
テストデータ件数: 8 | ラベル出現数: {'政治': 4, '経済': 5, 'スポーツ': 3, 'IT': 5}

4-fold CV で選ばれた閾値: best_th = 0.35


,閾値,訓練CV Micro-F1,訓練CV Samples-F1
0,0.20,0.609,0.594
1,0.25,0.607,0.594
2,0.30,0.624,0.610
3,0.35,0.661,0.655
4,0.40,0.660,0.659
5,0.45,0.625,0.617
6,0.50,0.559,0.517
7,0.55,0.370,0.340
8,0.60,0.213,0.167


---

## 2. 多クラス（Top-1）vs 多ラベル分類と評価指標の比較

テストデータ上で次の 3 手法を比較します。
1. **多クラス分類（Top-1：最大確率 1 ラベルのみ）**: 1 文書につき最も確率の高い 1 ラベルだけを出力した場合の振る舞い
2. **多ラベル分類（標準閾値 `0.50`）**: 各ラベル独立の 2 値分類器に既定の閾値 `0.50` を適用
3. **多ラベル分類（訓練 CV で選んだ閾値）**: 訓練 CV の Micro-F1 が最大となった `best_th` を適用

評価時には **Subset Accuracy（全ラベルが完全一致した割合）** は 1 つでも過不足があると 0 点になるため非常に厳しく、**Hamming Loss** や **Micro-F1 / Macro-F1 / Samples-F1** をあわせて確認します。

In [3]:
def eval_multilabel(name, Y_true, Y_pred):
    return {
        "手法": name,
        "Subset Accuracy": round(accuracy_score(Y_true, Y_pred), 3),
        "Hamming Loss": round(hamming_loss(Y_true, Y_pred), 3),
        "Micro-F1": round(f1_score(Y_true, Y_pred, average="micro", zero_division=0), 3),
        "Macro-F1": round(f1_score(Y_true, Y_pred, average="macro", zero_division=0), 3),
        "Samples-F1": round(f1_score(Y_true, Y_pred, average="samples", zero_division=0), 3),
    }

Y_top1 = np.zeros_like(Y_test)
Y_top1[np.arange(len(probs)), np.argmax(probs, axis=1)] = 1
Y_050 = (probs >= 0.50).astype(int)
Y_best = (probs >= best_th).astype(int)

summary_df = pd.DataFrame([
    eval_multilabel("多クラス分類（Top-1：最大確率 1 ラベルのみ）", Y_test, Y_top1),
    eval_multilabel("多ラベル分類（標準閾値 0.50）", Y_test, Y_050),
    eval_multilabel(f"多ラベル分類（訓練 CV で選んだ閾値 {best_th:.2f}）", Y_test, Y_best),
])
display(summary_df)

# 各見出しでの Top-1 予測 vs 訓練 CV 閾値での多ラベル予測と各ラベル確率
rows = []
for (txt, gold), p_row, top1_bin, best_bin in zip(test_samples, probs, Y_top1, Y_best):
    top1_lbls = [c for c, flag in zip(mlb.classes_, top1_bin) if flag]
    best_lbls = [c for c, flag in zip(mlb.classes_, best_bin) if flag]
    rows.append({
        "見出し": txt[:24] + "...",
        "正解ラベル": ",".join(gold),
        "Top-1予測": ",".join(top1_lbls),
        f"多ラベル予測(閾値{best_th:.2f})": ",".join(best_lbls),
        **{f"P({c})": round(float(p), 2) for c, p in zip(mlb.classes_, p_row)},
    })
pd.DataFrame(rows)

,手法,Subset Accuracy,Hamming Loss,Micro-F1,Macro-F1,Samples-F1
0,多クラス分類（Top-1：最大確率 1 ラベルのみ）,0.25,0.281,0.640,0.610,0.688
1,多ラベル分類（標準閾値 0.50）,0.25,0.219,0.741,0.735,0.762
2,多ラベル分類（訓練 CV で選んだ閾値 0.35）,0.50,0.125,0.889,0.904,0.857


,見出し,正解ラベル,Top-1予測,多ラベル予測(閾値0.35),P(政治),P(経済),P(スポーツ),P(IT)
0,元五輪選手が国会でスポーツ予算と経済効果について...,"政治,経済,スポーツ",政治,"政治,経済,スポーツ",0.66,0.43,0.50,0.25
1,政府がAIスタートアップへの補助金と規制緩和の新...,"政治,経済,IT",経済,"政治,経済,IT",0.50,0.58,0.27,0.47
2,国内IT大手がプロ野球球団の放映権ビジネスで増収...,"経済,スポーツ,IT",経済,"政治,経済,スポーツ,IT",0.37,0.54,0.52,0.35
3,サイバー攻撃対策の新法案が参議院本会議で可決され...,"政治,IT",政治,"政治,IT",0.66,0.28,0.30,0.44
4,半導体大手の四半期決算が過去最高益となり市場で買...,"経済,IT",経済,"経済,IT",0.31,0.62,0.29,0.44
5,プロ野球とサッカー日本代表の国際試合が国立競技場...,スポーツ,スポーツ,"経済,スポーツ",0.25,0.43,0.74,0.19
6,衆議院本会議で新年度予算案の採決と物価高対策が議...,"政治,経済",政治,政治,0.67,0.33,0.26,0.27
7,IT各社が新しい生成AI基盤モデルとクラウド開発...,IT,IT,"経済,IT",0.33,0.48,0.29,0.65


- **Top-1（単一ラベル強制）の限界**: 複合トピックの見出しでも 1 ラベルしか出力できないため、単一ラベルの見出し 2 件（`スポーツ`・`IT`）以外で必ず取りこぼしが生じ、Subset Accuracy は `0.250`、Micro-F1 は `0.640` に留まります。
- **標準閾値 `0.50` と訓練 CV 閾値 `0.35` の違い**: One-vs-Rest の各 2 値分類器では正例が少数派になりやすいため、副次ラベルの予測確率が `0.40〜0.48` 程度に留まる事例が多く、`0.50` では再現率が伸びません（Micro-F1 `0.741`）。テストデータを見ずに訓練データの 4 分割 CV だけで選んだ閾値 `0.35` を使うと、Micro-F1 が `0.889`（Samples-F1 `0.857`、Hamming Loss `0.125`）まで改善します。
- **閾値を下げるトレードオフ**: 一方で閾値を `0.35` に下げると、「国内IT大手がプロ野球球団の...」に `政治`（`P=0.37`）が追加されるなど偽陽性が増えるケースもあり、再現率と適合率のバランスを目的に応じて開発・CV データで調整する重要性がわかります。